# 3교시. 문서 구조 이해 및 추출 결과 정제

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/master/colab/03_document_structure.ipynb)

**이번 교시 행동:** 2교시 결과를 불러와 원문은 보존하고, 공백·날짜·표 영역만 정리합니다.

**통과 증거:** `course_outputs/clean_receipt.json`

> Google Colab도 외부 클라우드입니다. 조직 승인 없는 개인·회사 문서는
> 업로드하지 않습니다. 필수 실습은 저장소의 비식별 공개·합성 샘플만
> 사용합니다.

화면의 **처리 방식**을 먼저 확인합니다.

- **지금 이 사진을 직접 읽었습니다:** 현재 파일에 OCR 모델을 실행한 결과입니다.
- **수업용 예제 결과를 불러왔습니다:** 현재 파일을 분석한 결과가 아닙니다.
- 3분 이상 멈추면 실행을 중지하고 수업용 예제로 계속합니다.
- 각 교시 끝에서 `CHECKPOINT PASS`와 산출물 파일을 확인합니다.


## 이 노트북에서 내가 하는 일

- **필수 실습:** 2교시 OCR 결과의 낱말 좌표를 사람이 읽는 행과 문서 영역으로 다시 묶습니다.
- **내가 바꾸는 곳:** 품목 행을 찾는 정규식 한 줄만 채웁니다. 정답을 복사해 실행해도 됩니다.
- **인터넷 자료로 다시 실험:** 2교시에서 만든 다른 이미지의 `ocr_result.json`을 넣어 행 묶기가 어디서 깨지는지 비교합니다.

먼저 제공 샘플로 끝까지 실행해 `CHECKPOINT PASS`를 만드세요. 그다음
[공개·비식별 실습 자료 찾기](https://github.com/leecks1119/document_ai_lecture/blob/master/docs/public_practice_sources.md)를 보고
입력 한 장만 바꾸어 다시 실행합니다. 2교시에서 고른 자료와 결과 파일은
3~7교시에 그대로 이어 쓰므로 매 시간 새 자료를 찾을 필요가 없습니다.

> `🟢 그대로 실행하는 셀`은 수정하지 않습니다. `🟠 내가 짧게 바꾸는
> 셀`만 필수이고, `🔵 원하면 바꾸는 셀`은 시간이 남을 때 합니다.
> 정답은 모두 공개되어 있으므로 정답을 먼저 복사하고 결과를 관찰해도 됩니다.

## 코드 셀을 읽는 방법

각 코드 셀의 맨 위에는 `코드 읽기` 주석이 있습니다.

1. `수정하지 않습니다`라고 적힌 셀은 설명을 읽고 그대로 실행합니다.
2. 주황색 필수 `TODO`만 채웁니다. 파란색 선택 `TODO`는 건너뛰어도 됩니다.
3. 실행 출력에서 `코드 읽는 법`과 `확인할 결과`를 다시 확인합니다.
4. `단계 실행 완료`가 나온 뒤 다음 코드 셀로 이동합니다.

Python 문법 전체를 먼저 이해할 필요는 없습니다. 변수에 어떤 값이 들어가고,
실행 뒤 어떤 결과가 달라지는지를 중심으로 읽습니다.


In [ ]:
def _show_learning_message(markdown_text):
    try:
        from IPython.display import Markdown, display
        display(Markdown(markdown_text))
    except ImportError:
        print(markdown_text)


def show_lab_step(
    current,
    total,
    title,
    action,
    expected,
    code_help,
    edit_kind,
):
    cell_kind = {
        "required": "🟠 내가 짧게 바꾸는 셀",
        "optional": "🔵 원하면 바꾸는 셀",
        "none": "🟢 그대로 실행하는 셀",
    }[edit_kind]
    _show_learning_message(
        f"""---
### {cell_kind} · {current}/{total} · {title}

**지금 할 일:** {action}

**코드 읽는 법:** {code_help}

**이 단계에서 확인할 결과:** {expected}
"""
    )


def complete_lab_step(current, total, expected):
    next_action = (
        "결과를 확인한 뒤 다음 코드 셀을 실행하세요."
        if current < total
        else "마지막 CHECKPOINT와 산출물 파일을 확인하세요."
    )
    _show_learning_message(
        f"""> ✅ **{current}/{total} 단계 실행 완료**
>
> **결과 확인:** {expected}
>
> **다음 행동:** {next_action}
"""
    )

# ── 코드 읽기 ─────────────────────────────────────────────
# 이전 교시의 `ocr_result.json`을 받을 공통 폴더와 업로드 함수를 준비합니다. 설정 코드이므로 수정하지 않습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(1, 6, '공통 환경 준비', '이전 교시 산출물을 받을 폴더와 복구 기능을 준비합니다.', 'Python·Platform·공통 작업 폴더가 표시되어야 합니다.', '이전 교시의 `ocr_result.json`을 받을 공통 폴더와 업로드 함수를 준비합니다. 설정 코드이므로 수정하지 않습니다.', 'none')

import json
import os
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
VALIDATION_MODE = os.getenv("COURSE_VALIDATE_EXAMPLE") == "1"

def upload_previous_artifact(filename):
    target = OUTPUT_DIR / filename
    if target.exists() or VALIDATION_MODE:
        return target if target.exists() else None
    try:
        from google.colab import files
    except ImportError:
        return None
    print(f"이전 교시에서 내려받은 {filename}을 선택하세요.")
    uploaded = files.upload()
    if filename not in uploaded:
        raise FileNotFoundError(
            f"{filename}이 선택되지 않았습니다. 준비 입력을 쓰려면 "
            "USE_COURSE_EXAMPLE=True로 바꾸세요."
        )
    target.write_bytes(uploaded[filename])
    return target


def download_artifact(path):
    if VALIDATION_MODE:
        return
    try:
        from google.colab import files
    except ImportError:
        return
    files.download(str(path))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("공통 작업 폴더:", OUTPUT_DIR.resolve())

COURSE_ASSET_BASE_URL = (
    "https://raw.githubusercontent.com/leecks1119/"
    "document_ai_lecture/master/"
)

def load_course_assets(*relative_paths):
    if VALIDATION_MODE:
        local_root = os.getenv("COURSE_LOCAL_ASSET_ROOT")
        if not local_root:
            raise RuntimeError(
                "자동 검증용 COURSE_LOCAL_ASSET_ROOT가 필요합니다."
            )
        root = Path(local_root)
        return {
            path: (root / path).read_bytes()
            for path in relative_paths
        }

    import requests

    loaded = {}
    missing = []
    for path in relative_paths:
        try:
            response = requests.get(
                COURSE_ASSET_BASE_URL + path,
                timeout=30,
            )
            response.raise_for_status()
            loaded[path] = response.content
        except requests.RequestException as exc:
            print(f"자동 다운로드 실패: {Path(path).name} · {exc}")
            missing.append(path)

    if missing:
        from google.colab import files

        expected = ", ".join(Path(path).name for path in missing)
        print("다음 파일을 저장소에서 내려받아 선택하세요:", expected)
        uploaded = files.upload()
        uploaded_by_name = {
            Path(name).name: content
            for name, content in uploaded.items()
        }
        for path in missing:
            filename = Path(path).name
            if filename not in uploaded_by_name:
                raise FileNotFoundError(
                    f"{filename}이 선택되지 않았습니다."
                )
            loaded[path] = uploaded_by_name[filename]

    return loaded

complete_lab_step(1, 6, 'Python·Platform·공통 작업 폴더가 표시되어야 합니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `GOLDEN_OCR_TEXT`와 `GOLDEN_RECEIPT`는 파일 인계가 막힐 때만 쓰는 공개 복구 데이터입니다. 수업용 예제와 지금 실행한
# 결과를 구분합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(2, 6, '준비 입력 등록', '이전 산출물이 없을 때 사용할 공개 OCR 정답을 준비합니다.', '오류 없이 끝나면 준비 입력이 메모리에 등록된 것입니다.', '`GOLDEN_OCR_TEXT`와 `GOLDEN_RECEIPT`는 파일 인계가 막힐 때만 쓰는 공개 복구 데이터입니다. 수업용 예제와 지금 실행한 결과를 구분합니다.', 'none')

GOLDEN_OCR_TEXT = '이태리집\n거래일시 2025-10-04 12:33:37\n페퍼로니 앤 치즈 29,000 1 29,000\n토마토 파스타 14,000 1 14,000\n수제 돈가스 13,000 1 13,000\n새우 칠리치 필라 14,000 1 14,000\n콜라 2,000 3 6,000\n합계 금액 76,000\n부가세 과세물품가액 69,094\n부가세 6,906\n'
GOLDEN_VLM_MARKDOWN = '# 이태리집\n\n> **수업용 VLM 구조 예제** — 지금 모델을 실행해 만든 결과가 아닙니다.\n\n거래일시: 2025-10-04 12:33:37\n\n| 품목 | 수량 | 단가 | 금액 |\n| --- | ---: | ---: | ---: |\n| 페퍼로니 앤 치즈 | 1 | 29,000원 | 29,000원 |\n| 토마토 파스타 | 1 | 14,000원 | 14,000원 |\n| 수제 돈가스 | 1 | 13,000원 | 13,000원 |\n| 새우 칠리치 필라 | 1 | 14,000원 | 14,000원 |\n| 콜라 | 3 | 2,000원 | 6,000원 |\n\n**합계: 76,000원**\n\n부가세 과세물품가액 69,094\n부가세 6,906\n'
GOLDEN_RECEIPT = {'document_type': 'receipt',
 'store_name': '이태리집',
 'date': '2025-10-04',
 'total_amount': 76000,
 'items': [{'name': '페퍼로니 앤 치즈',
            'quantity': 1,
            'unit_price': 29000,
            'line_total': 29000},
           {'name': '토마토 파스타', 'quantity': 1, 'unit_price': 14000, 'line_total': 14000},
           {'name': '수제 돈가스', 'quantity': 1, 'unit_price': 13000, 'line_total': 13000},
           {'name': '새우 칠리치 필라',
            'quantity': 1,
            'unit_price': 14000,
            'line_total': 14000},
           {'name': '콜라', 'quantity': 3, 'unit_price': 2000, 'line_total': 6000}],
 'adjustments': {'discount': 0, 'tax': 0, 'service': 0, 'rounding': 0},
 'tax_breakdown': {'mode': 'included_in_item_prices',
                   'supply_amount': 69094,
                   'vat': 6906,
                   'payable_total': 76000},
 'raw_values': {'store_name': '이태리집',
                'date': '2025-10-04 12:33:37',
                'total_amount': '76,000'},
 'cleaned_values': {'store_name': '이태리집', 'date': '2025-10-04', 'total_amount': 76000},
 'evidence': {'store_name': {'raw_value': '이태리집', 'line': 1},
              'date': {'raw_value': '거래일시 2025-10-04 12:33:37', 'line': 2},
              'total_amount': {'raw_value': '합계 금액 76,000', 'line': 8}},
 'source_mode': 'course_example_rule_extraction'}

complete_lab_step(2, 6, '오류 없이 끝나면 준비 입력이 메모리에 등록된 것입니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `reconstruct_spatial_lines()`는 y좌표가 가까운 토큰을 같은 행으로 묶고 x좌표 순서로 정렬합니다. OCR 글자를 읽기
# 순서로 복원하는 함수입니다.
# ──────────────────────────────────────────────────────────
show_lab_step(3, 6, '공간 순서 복원 함수 준비', 'OCR 좌표를 행과 읽기 순서로 재구성하는 함수를 만듭니다.', '오류 없이 끝나면 재구성 함수를 사용할 수 있습니다.', '`reconstruct_spatial_lines()`는 y좌표가 가까운 토큰을 같은 행으로 묶고 x좌표 순서로 정렬합니다. OCR 글자를 읽기 순서로 복원하는 함수입니다.', 'none')

from collections import defaultdict

def reconstruct_spatial_lines(items):
    positioned_by_page = defaultdict(list)
    unpositioned_by_page = defaultdict(list)
    for order, item in enumerate(items):
        text = " ".join(str(item.get("text", "")).split())
        if not text:
            continue
        page = int(item.get("page") or 1)
        points = [
            point
            for point in (item.get("box") or [])
            if isinstance(point, (list, tuple)) and len(point) >= 2
        ]
        if not points:
            unpositioned_by_page[page].append((order, text))
            continue
        xs = [float(point[0]) for point in points]
        ys = [float(point[1]) for point in points]
        positioned_by_page[page].append({
            "text": text,
            "x": min(xs),
            "y": sum(ys) / len(ys),
            "height": max(ys) - min(ys),
            "order": order,
        })

    pages = sorted(set(positioned_by_page) | set(unpositioned_by_page))
    lines = []
    for page in pages:
        rows = []
        for token in sorted(
            positioned_by_page[page],
            key=lambda value: (value["y"], value["x"], value["order"]),
        ):
            row = rows[-1] if rows else None
            tolerance = (
                max(12.0, min(24.0, max(row["height"], token["height"]) * 0.45))
                if row else 12.0
            )
            if row and abs(token["y"] - row["y"]) <= tolerance:
                row["tokens"].append(token)
                count = len(row["tokens"])
                row["y"] = (row["y"] * (count - 1) + token["y"]) / count
                row["height"] = max(row["height"], token["height"])
            else:
                rows.append({
                    "tokens": [token],
                    "y": token["y"],
                    "height": token["height"],
                })

        lines.extend(
            " ".join(
                token["text"]
                for token in sorted(
                    row["tokens"],
                    key=lambda value: (value["x"], value["order"]),
                )
            )
            for row in rows
        )
        lines.extend(
            text
            for _, text in sorted(
                unpositioned_by_page[page],
                key=lambda value: value[0],
            )
        )
    return lines

complete_lab_step(3, 6, '오류 없이 끝나면 재구성 함수를 사용할 수 있습니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `USE_COURSE_EXAMPLE=True`면 새 Colab에서 공개 입력을 씁니다. 앞 교시 파일을 이어 쓰려면 `False`로 바꾸며,
# `groups`가 문서 영역을 나눕니다.
# ──────────────────────────────────────────────────────────
show_lab_step(4, 6, 'OCR를 문서 구조로 변환', '이전 OCR 결과를 읽고 헤더·품목·합계 후보로 나눕니다.', '입력 모드·원문 줄·품목 후보 수와 JSON 경로를 확인합니다.', '`USE_COURSE_EXAMPLE=True`면 새 Colab에서 공개 입력을 씁니다. 앞 교시 파일을 이어 쓰려면 `False`로 바꾸며, `groups`가 문서 영역을 나눕니다.', 'none')

import re

previous_path = OUTPUT_DIR / "ocr_result.json"
# 기본값 True: 새 Colab에서도 공개 준비 입력으로 바로 실행합니다.
# 앞 교시 파일을 이어 쓰려면 False로 바꾸고 업로드 창에서 선택합니다.
USE_COURSE_EXAMPLE = True
if not previous_path.exists() and not USE_COURSE_EXAMPLE:
    upload_previous_artifact("ocr_result.json")
if previous_path.exists():
    previous = json.loads(previous_path.read_text(encoding="utf-8"))
    raw_text = "\n".join(item["text"] for item in previous["items"])
    layout_lines = reconstruct_spatial_lines(previous["items"])
    INPUT_MODE = "PREVIOUS_LESSON"
else:
    raw_text = GOLDEN_OCR_TEXT
    layout_lines = raw_text.splitlines()
    INPUT_MODE = "COURSE_EXAMPLE"
print(
    "입력 자료:",
    (
        "2교시에서 만든 OCR 결과를 불러왔습니다."
        if INPUT_MODE == "PREVIOUS_LESSON"
        else "수업용 예제 OCR 결과를 불러왔습니다."
    ),
)
if INPUT_MODE == "COURSE_EXAMPLE":
    print("중요: 지금 새로 OCR을 실행한 결과가 아닙니다.")


def clean_lines(text, source_lines):
    cleaned = []
    changes = []
    for raw in source_lines:
        normalized = re.sub(r"\s+", " ", raw.strip())
        if normalized:
            cleaned.append(normalized)
        if raw != normalized:
            changes.append({"before": raw, "after": normalized})
    groups = {"header": [], "date": [], "items": [], "total": [], "other": []}
    for line in cleaned:
        if re.search(r"\d{4}[-./]\d{1,2}[-./]\d{1,2}", line):
            groups["date"].append(line)
        elif "합계" in line:
            groups["total"].append(line)
        elif re.search(r"[\d,]+\s+\d+\s+[\d,]+$", line):
            groups["items"].append(line)
        elif not groups["header"]:
            groups["header"].append(line)
        else:
            groups["other"].append(line)
    return {
        "input_mode": INPUT_MODE,
        "raw_text": text,
        "layout_lines": source_lines,
        "cleaned_lines": cleaned,
        "groups": groups,
        "change_log": changes,
        "rule": "원문에 없는 값은 추가하지 않음",
    }


clean_result = clean_lines(raw_text, layout_lines)
output_path = OUTPUT_DIR / "clean_receipt.json"
output_path.write_text(
    json.dumps(clean_result, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)
assert clean_result["raw_text"] == raw_text
print("원문 줄:", len(raw_text.splitlines()))
print("품목 후보 줄:", len(clean_result["groups"]["items"]))
print("CHECKPOINT 1/1 PASS:", output_path)
download_artifact(output_path)

complete_lab_step(4, 6, '입력 모드·원문 줄·품목 후보 수와 JSON 경로를 확인합니다.')


## 내가 직접 채우는 1줄

품목 행은 끝부분에 `단가 수량 금액`이 반복됩니다. 그 모양을 찾는
정규식 한 줄을 채웁니다.


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `my_item_rule` 한 곳만 정규식으로 채웁니다. 숫자 세 묶음으로 끝나는 행을 품목 후보로 찾는 규칙입니다.
# ──────────────────────────────────────────────────────────
show_lab_step(5, 6, '내 품목 행 규칙 입력', '품목 행 끝의 단가·수량·금액 패턴을 정규식으로 표현합니다.', '내 규칙으로 찾은 품목 후보 수가 표시되어야 합니다.', '`my_item_rule` 한 곳만 정규식으로 채웁니다. 숫자 세 묶음으로 끝나는 행을 품목 후보로 찾는 규칙입니다.', 'required')

# TODO: 품목 행 끝의 숫자 세 묶음을 찾는 정규식을 넣으세요.
my_item_rule = None
if my_item_rule is None:
    print("빈칸입니다. 아래 전체 정답과 비교하세요.")

complete_lab_step(5, 6, '내 규칙으로 찾은 품목 후보 수가 표시되어야 합니다.')


<details>
<summary>힌트와 전체 정답 보기</summary>

숫자·쉼표 묶음, 수량 정수, 마지막 숫자·쉼표 묶음을 공백으로 연결합니다.
</details>


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `ANSWER_ITEM_RULE`은 공개 정규식입니다. `re.search()`가 각 행에서 규칙과 일치하는 품목 다섯 개를 찾는지 확인합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(6, 6, '품목 행 규칙 정답 확인', '공개 정규식으로 다섯 품목 행을 다시 찾습니다.', '다섯 품목 문자열이 목록으로 표시되어야 합니다.', '`ANSWER_ITEM_RULE`은 공개 정규식입니다. `re.search()`가 각 행에서 규칙과 일치하는 품목 다섯 개를 찾는지 확인합니다.', 'none')

ANSWER_ITEM_RULE = r"[\d,]+\s+\d+\s+[\d,]+$"
answer_item_lines = [
    line
    for line in clean_result["cleaned_lines"]
    if re.search(ANSWER_ITEM_RULE, line)
]
assert answer_item_lines
print("전체 정답 · 찾은 품목 후보:", answer_item_lines)

complete_lab_step(6, 6, '다섯 품목 문자열이 목록으로 표시되어야 합니다.')
